# Quality Control of Processed Single-Cell Features

This notebook performs quality control after single-cell feature processing. The processed table has already undergone area filtering, 99th percentile marker censoring, and arcsinh transformation with cofactor 1.

Purpose of this QC step:

- Confirm that the processed single-cell table exists and has the expected structure.
- Summarize how many cells are present per image.
- Check the area distribution after filtering.
- Check marker value ranges after censoring and arcsinh transformation.
- Create simple figures that make the processed data easier to inspect before phenotyping.

This step is a technical quality-control step. It does not assign cell phenotypes and does not test biological hypotheses.


## Step 0: Configure Workflow Paths

The setup cell defines the workflow folders and expected input/output files. The notebook can be executed from the workflow root or from inside the `notebooks/` directory.

Main input:

```text
results/11_processed_single_cell_features.csv
```

Main outputs:

```text
results/12_processed_single_cell_qc_summary.csv
results/12_processed_marker_qc_summary.csv
figures/01_cells_per_image.png
figures/02_area_distribution_after_filtering.png
figures/03_marker_summary_after_transformation.png
```


In [ ]:
from pathlib import Path
import csv
import subprocess

current = Path.cwd().resolve()
if current.name == 'notebooks':
    WORKFLOW_DIR = current.parent
else:
    WORKFLOW_DIR = current

RESULTS_DIR = WORKFLOW_DIR / 'results'
FIGURES_DIR = WORKFLOW_DIR / 'figures'
SCRIPTS_DIR = WORKFLOW_DIR / 'scripts'

INPUT = RESULTS_DIR / '11_processed_single_cell_features.csv'
SUMMARY = RESULTS_DIR / '12_processed_single_cell_qc_summary.csv'
MARKER_SUMMARY = RESULTS_DIR / '12_processed_marker_qc_summary.csv'

print('Workflow directory:', WORKFLOW_DIR)
print('Processed feature table exists:', INPUT.exists())
print('QC script exists:', (SCRIPTS_DIR / '09_qc_processed_single_cell_features.py').exists())


## Step 1: Run QC Summary and Figure Generation

The script below reads the processed single-cell table and marker panel. Marker columns are detected from `data/panel.csv`, which prevents morphology or coordinate columns from being summarized as marker channels.

QC outputs generated by this step:

- Overall summary table: total cells, image count, area statistics, and cells per image.
- Marker summary table: minimum, median, 99th percentile, maximum, and invalid-value count for each transformed marker.
- Cell count bar plot: verifies that each image contributed cells.
- Area histogram: verifies that very small objects were removed and shows the remaining area distribution.
- Marker summary plot: compares marker medians and high-end transformed values after processing.


In [ ]:
cmd = [
    'python3',
    str(SCRIPTS_DIR / '09_qc_processed_single_cell_features.py'),
    '--workflow-dir', str(WORKFLOW_DIR),
    '--input', 'results/11_processed_single_cell_features.csv',
    '--panel', 'data/panel.csv',
    '--summary-output', 'results/12_processed_single_cell_qc_summary.csv',
    '--marker-output', 'results/12_processed_marker_qc_summary.csv',
    '--cell-count-figure', 'figures/01_cells_per_image.png',
    '--area-figure', 'figures/02_area_distribution_after_filtering.png',
    '--marker-figure', 'figures/03_marker_summary_after_transformation.png',
]

result = subprocess.run(cmd, cwd=WORKFLOW_DIR, text=True, capture_output=True, check=True)
print(result.stdout)


## Step 2: Review Overall QC Summary

The overall QC summary answers basic file-level and image-level questions:

- How many cells remain after processing?
- How many images are represented?
- How many marker columns are present?
- What is the minimum area after filtering?
- How many cells came from each image?

A minimum area of at least 4 pixels confirms that the template-paper area filter was applied. Very uneven cell counts across images may indicate differences in tissue content, segmentation behavior, or image quality.


In [ ]:
with SUMMARY.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    summary_rows = list(reader)

for row in summary_rows[:20]:
    print(row)


## Step 3: Review Marker QC Summary

The marker QC summary checks transformed marker columns after censoring and arcsinh transformation.

Important columns:

- `valid_values`: number of usable numeric values for the marker.
- `missing_or_invalid_values`: number of missing, non-numeric, infinite, or NaN values.
- `minimum`: smallest transformed marker value.
- `median`: typical transformed marker value.
- `p99`: high-end transformed marker value after processing.
- `maximum`: largest transformed marker value after processing.

After 99th percentile censoring and arcsinh transformation, marker maximum values should be compressed compared with raw exported intensities. Invalid-value counts should ideally be zero.


In [ ]:
with MARKER_SUMMARY.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    marker_rows = list(reader)

for row in marker_rows[:15]:
    print(row)


## Step 4: Review Generated Figures

The generated figures provide a visual QC check before phenotyping or spatial interpretation.

Figure interpretation:

- `01_cells_per_image.png`: detects missing images or unusually low/high cell counts.
- `02_area_distribution_after_filtering.png`: confirms the object-size distribution after area filtering.
- `03_marker_summary_after_transformation.png`: summarizes typical and high-end transformed marker values across the panel.


In [ ]:
expected_figures = [
    FIGURES_DIR / '01_cells_per_image.png',
    FIGURES_DIR / '02_area_distribution_after_filtering.png',
    FIGURES_DIR / '03_marker_summary_after_transformation.png',
]

for figure in expected_figures:
    print(figure.name, 'exists:', figure.exists(), 'size_bytes:', figure.stat().st_size if figure.exists() else 0)
